In [1]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 8-1 Introduccion

##### 8-2 Cuando usar SQL

##### 8-3 Cuando usar ETL

##### 8-4 Cuando usar otras herramientas

##### 8.5 Comentar código

##### 8.6 Estilo

##### 8.7 Almacenamiento de código

##### 8-8 Orden de Evaluacion


Se ejecutan en el siguiente orden:

- FROM (incluyendo JOINS en su clausula ON)

- WHERE

- GROUP BY

- HAVING

- WINDOW FUNCTION

- SELECT

- DISTINCT

- UNION

- ORDER BY

- LIMIT y OFFSET

##### 8-9 Tablas Temporales

Las tablas temporales son tablas que pueden ser creadas y existiran unicamente durante la sesión actual y se eliminaran automaticamente cuando la sesión finalice.

Para crear una tabla temporal se requiere permiso de escritura en la base de datos 


In [26]:
# Crear un esquema (borralo antes si ya existe)

query1 = r"""
DROP DATABASE IF EXISTS new_schema;
"""

query2 = r"""
CREATE DATABASE new_schema;
"""

with engine.connect() as connection:
    connection.execute(text(query1))    
    connection.execute(text(query2))
    connection.commit()


In [ ]:
# Crear una tabla temporal */




query1 = r"""
CREATE TEMPORARY TABLE new_schema.T_ciudades(
ciudad_id	INT PRIMARY KEY AUTO_INCREMENT,
ciudad		VARCHAR(25) 		
);
"""

with engine.begin() as conn:
    conn.execute(text(query1)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    connection.commit()


In [38]:
# Si es necesario podemos eliminar la tabla temporal

#query = r"""
#DROP TEMPORARY TABLE new_schema.T_ciudades;
#"""
#with engine.begin() as conn:
#    conn.execute(text(query)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
#    connection.commit()

In [39]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad_id,ciudad


In [40]:
# Insertar filas a la tabla temporal */

query = r"""
INSERT INTO new_schema.T_ciudades(ciudad)
SELECT 	distinct ciudad 
FROM 	cohort.supertienda;
"""

with engine.begin() as conn:
    conn.execute(text(query)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    connection.commit()

In [41]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad_id,ciudad
0,1,Houston
1,2,Naperville
2,3,Philadelphia
3,4,Henderson
4,5,Athens


In [50]:
# Se puede crear y poblar la tabla temporal en una sola sentencia */

# Borrar la tabla temporal si ya existe
query1 = r"""
DROP TABLE new_schema.T_ciudades;
"""

# Crear la tabla temporal a partir de una consulta
query2 = r"""
CREATE TEMPORARY TABLE new_schema.T_ciudades AS
SELECT distinct ciudad
FROM cohort.supertienda;
"""
with engine.begin() as conn:
    conn.execute(text(query1))
    conn.execute(text(query2))
    connection.commit()

In [51]:
# Visualizar la tabla temporal

query = r"""
select
    * 
from
    new_schema.T_ciudades;
"""
df = pd.read_sql_query(query, engine)
df.head()

,ciudad
0,Houston
1,Naperville
2,Philadelphia
3,Henderson
4,Athens


##### 8-10 CTE: Expresiones de Tabla Comunes

Una CTE (Common Table Expression o Expresión de Tabla Común) es un conjunto de resultados temporales con nombre que se define al inicio de una consulta usando la cláusula WITH.

Funciona como una tabla virtual o vista que solo existe mientras se está ejecutando la instrucción principal (SELECT, INSERT, UPDATE o DELETE) y no se guarda de forma permanente en la base de datos. 

Están disponibles en MySQL a partir de la versión 8.0.

In [61]:
# Veamos la tabla supertienda

query = r"""
select *
from cohort.supertienda;
"""
df = pd.read_sql_query(text(query), engine)
df.head(5)

,venta_id,Categoria,Ciudad,Pais/Región,Nombre_del_Cliente,Proveedor,Fecha_de_Orden,ID_Orden,Codigo_Postal,Nombre_de_Producto,Region,Segmento,Fecha_de_envio,Estado/Provincia,Descuento,Ganancia,Cantidad,Ventas
0,1,Suministros de Oficina,Houston,United States,Darren Powers,Message Book,2019-01-03,US-2019-103800,77095,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Central,Consumidor,2019-01-07,Texas,0.2,5.5512,2,16.448
1,2,Suministros de Oficina,Naperville,United States,Phillina Ober,GBC,2019-01-04,US-2019-112326,60540,GBC Standard Plastic Binding Systems Combs,Central,Home Office,2019-01-08,Illinois,0.8,-5.4870,2,3.540
2,3,Suministros de Oficina,Naperville,United States,Phillina Ober,Avery,2019-01-04,US-2019-112326,60540,Avery 508,Central,Home Office,2019-01-08,Illinois,0.2,4.2717,3,11.784
3,4,Suministros de Oficina,Naperville,United States,Phillina Ober,SAFCO,2019-01-04,US-2019-112326,60540,SAFCO Boltless Steel Shelving,Central,Home Office,2019-01-08,Illinois,0.2,-64.7748,3,272.736
4,5,Suministros de Oficina,Philadelphia,United States,Mick Brown,Avery,2019-01-05,US-2019-141817,19143,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Este,Consumidor,2019-01-12,Pennsylvania,0.2,4.8840,3,19.536


In [65]:
# Usarmos un CTE (Common Table Expression) para calcular el numero de clientes retenidos por año.

query = r"""
WITH cte_primer_compra as
(
SELECT
    nombre_del_cliente,
    min(fecha_de_orden) as primer_compra
FROM
    cohort.supertienda
GROUP BY
    1
)
SELECT
    timestampdiff(year,a.primer_compra,b.fecha_de_orden) as periodos,
    count(distinct(b.nombre_del_cliente)) as cohorte_retenida
FROM
    cte_primer_compra as a JOIN cohort.supertienda as b on a.nombre_del_cliente = b.nombre_del_cliente
GROUP BY 
    1
;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,periodos,cohorte_retenida
0,0,800
1,1,619
2,2,588
3,3,403
